# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR² Dataset](https://doi.org/10.71728/senscience.qs2f-h81p) using the `mlcroissant` library.

### Dataset Source
The dataset is specified via a Croissant schema URL and follows the [MLCommons Croissant data packaging standard](https://mlcommons.github.io/croissant/).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List the available record sets and their fields using their `@id`. This will help you identify which tables and fields are available for extraction and analysis.

In [ ]:
# Inspect record sets (@id and name/title)
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in the Croissant schema.")
else:
    print("Available record sets:\n---------------------")
    for rs in record_sets:
        print(f"@id: {rs['@id']}")
        print(f"  name: {rs.get('name', '(no name)')}")
        # List fields in this record set
        if 'field' in rs:
            fields = rs['field']
            if not isinstance(fields, list):
                fields = [fields]
            print("  Fields and their @id:")
            for field in fields:
                if isinstance(field, dict):
                    print(f"    - {field.get('@id', '(no id)')}")
                else:
                    print(f"    - {field}")
        print("")

## 3. Data Extraction
Load the primary clinical records record set (use its `@id` as shown above) into a DataFrame for analysis. You can specify additional record sets by `@id` if needed.

In [ ]:
# ---
# Identify primary record set (by inspection from previous cell):
# If you printed a list of record set @ids, select the main clinical data table.

# For this dataset, the main record set is very likely to be:
rcset_ids = [rs['@id'] for rs in dataset.record_sets] if dataset.record_sets else []
print("Record sets available:")
for i, rid in enumerate(rcset_ids):
    print(f"{i}: {rid}")
# Pick the first available record set if only one is present
main_rcset_id = rcset_ids[0] if rcset_ids else None
print(f"\nSelected record set for extraction: {main_rcset_id}")

dataframes = {}
if main_rcset_id:
    records = list(dataset.records(record_set=main_rcset_id))
    dataframes[main_rcset_id] = pd.DataFrame(records)
    print("\nColumns in main record set:")
    print(dataframes[main_rcset_id].columns.tolist())
    display(dataframes[main_rcset_id].head())
else:
    print("No record set to extract.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. 

Below, select some fields by their `@id` (from previous overview) for demonstration.

In [ ]:
# ---
# Example EDA: Filter and normalize one numeric field (edit IDs as found in your exploration above!)

# For demonstration, let's try to guess a numeric field from the columns (e.g., 'Age', 'IntervalBetweenDiagnoses', or similar). Adjust as needed!
df = dataframes[main_rcset_id]
numeric_candidates = [col for col in df.columns if any(kw in col.lower() for kw in ["age", "interval", "time", "year", "duration", "months"])]

if numeric_candidates:
    numeric_field = numeric_candidates[0]
    print(f"Using numeric field for analysis: {numeric_field}")
    # Convert to number (if necessary)
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
else:
    print("No obvious numeric fields detected. Please update 'numeric_field' manually to a relevant column.")
    numeric_field = df.columns[0]  # fallback

# Filtering
threshold = df[numeric_field].quantile(0.70) if pd.api.types.is_numeric_dtype(df[numeric_field]) else None
if threshold is not None:
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (top 30%):")
else:
    filtered_df = df.copy()

display(filtered_df.head())

# Normalization
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(f"Normalized {numeric_field}:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
else:
    print(f"{numeric_field} is not numeric, skipping normalization.")

# Grouping (by e.g., 'Sex' or 'AnatomicalLocation', adjust with @id or column name as needed)
possible_groups = [col for col in df.columns if any(kw in col.lower() for kw in ["sex", "anatomical", "site", "location", "status"])]
group_field = possible_groups[0] if possible_groups else None

if group_field and pd.api.types.is_numeric_dtype(df[numeric_field]):
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped mean {numeric_field} by {group_field}:")
    display(grouped_df.head())
else:
    print("No group field found or data is non-numeric for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the main numeric field
if pd.api.types.is_numeric_dtype(df[numeric_field]):
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

# Boxplot by group field
if group_field and pd.api.types.is_numeric_dtype(df[numeric_field]):
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
In this notebook, we explored and processed the FAIR² clinical dataset using the `mlcroissant` library. We loaded metadata and clinical records, reviewed available fields by their `@id`, performed filtering, normalization, and summary statistics, and visualized the primary quantitative attributes stratified by relevant groupings such as anatomical site or patient sex. This workflow can easily be extended for more advanced data analysis or modeling, leveraging the structured metadata provided by the Croissant schema.